# Sentiment Forecasting with FinBERT and Topological Data Analysis

This notebook walks through the full pipeline: fetching financial news, scoring sentiment with FinBERT, computing topological features from price data, and backtesting a simple trading strategy.

**Some points regarding the project:**
- FinBERT is specifically trained on financial text, so it handles market jargon better than generic sentiment models
- Topological Data Analysis (TDA) looks at the "shape" of price movements using tools from algebraic topology
- Everything runs end-to-end with proper time-series validation (no lookahead bias)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TICKERS
from src.news import fetch_all_headlines
from src.sentiment import score_headlines, aggregate_daily_sentiment
from src.features import compute_price_features, join_sentiment_with_prices
from src.ml import add_quick_prob, train_model_from_frame, add_ml_prob
from src.backtest import apply_signals, backtest_equal_weight, todays_signals

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

## 1. Fetching Headlines

We pull recent headlines from Google News RSS and yfinance. The pipeline tries Google first and falls back to yfinance if needed.

In [ ]:
tickers = ['SPY', 'QQQ', 'AAPL', 'NVDA', 'MSFT']
news = fetch_all_headlines(tickers, days=7)

print(f"Fetched {len(news)} headlines")
news.sample(5)[['ticker', 'headline', 'ts']]

## 2. Sentiment Scoring with FinBERT

FinBERT classifies each headline as positive, negative, or neutral. The model outputs a confidence score, which we convert to a signed value: positive headlines get +confidence, negative get -confidence, neutral stays at 0.

In [ ]:
scored = score_headlines(news)
scored[['ticker', 'headline', 'sent_label', 'sent_score']].head(10)

In [ ]:
# Quick look at the distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

scored['sent_label'].value_counts().plot.bar(ax=axes[0], color=['#4CAF50', '#F44336', '#9E9E9E'])
axes[0].set_title('Sentiment Labels')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

axes[1].hist(scored['sent_score'], bins=30, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Sentiment Score Distribution')
axes[1].set_xlabel('Score (-1 to +1)')

plt.tight_layout()
plt.show()

In [ ]:
# Average sentiment by ticker
ticker_sent = scored.groupby('ticker')['sent_score'].agg(['mean', 'count'])
ticker_sent.columns = ['avg_sentiment', 'num_headlines']
ticker_sent.sort_values('avg_sentiment', ascending=False)

## 3. Topological Data Analysis

TDA applies concepts from algebraic topology to find patterns in data.

The basic idea:
1. Take a window of returns (say 60 days)
2. Use **delay embedding** to convert the 1D time series into a point cloud in higher dimensions
3. Compute **persistent homology**. This tracks topological features (clusters, loops) as we vary a scale parameter
4. Extract summary statistics from the persistence diagram

The intuition is that different market regimes (trending, mean-reverting, choppy) produce different topological signatures.

In [ ]:
# Compute price features including topology
price_features = compute_price_features(tickers, lookback_days=180, add_topology=True)

print(f"Shape: {price_features.shape}")
print(f"\nColumns: {list(price_features.columns)}")

In [ ]:
# Visualize topology features for SPY
spy = price_features[price_features['ticker'] == 'SPY'].copy()
spy['date'] = pd.to_datetime(spy['date'])

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].bar(spy['date'], spy['ret_5'], color=np.where(spy['ret_5'] > 0, 'green', 'red'), alpha=0.7)
axes[0].set_ylabel('5-day return')
axes[0].set_title('SPY: Returns and Topology Features')

axes[1].plot(spy['date'], spy['topo_d0_total'], color='steelblue')
axes[1].fill_between(spy['date'], spy['topo_d0_total'], alpha=0.3)
axes[1].set_ylabel('H0 persistence')

axes[2].plot(spy['date'], spy['topo_entropy'], color='darkorange')
axes[2].fill_between(spy['date'], spy['topo_entropy'], alpha=0.3, color='orange')
axes[2].set_ylabel('Topo entropy')

plt.tight_layout()
plt.show()

### Quick primer on delay embedding

If you have a time series $r_1, r_2, ..., r_n$, a delay embedding with dimension 3 creates points:
- $(r_1, r_2, r_3)$
- $(r_2, r_3, r_4)$
- ...

This transforms a 1D series into a point cloud. TDA then analyzes the shape of this cloud.

In [ ]:
# Visualize the embedding for a sample window
returns = spy['ret_1'].dropna().tail(60).values

# Create 3D embedding
embedded = np.array([[returns[i], returns[i+1], returns[i+2]] 
                     for i in range(len(returns)-2)])

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Color by time
colors = plt.cm.viridis(np.linspace(0, 1, len(embedded)))
ax.scatter(embedded[:, 0], embedded[:, 1], embedded[:, 2], c=colors, s=30)

# Connect consecutive points
for i in range(len(embedded)-1):
    ax.plot3D(embedded[i:i+2, 0], embedded[i:i+2, 1], embedded[i:i+2, 2], 
              color='gray', alpha=0.3, linewidth=0.5)

ax.set_xlabel('r(t)')
ax.set_ylabel('r(t-1)')
ax.set_zlabel('r(t-2)')
ax.set_title('Delay Embedding of SPY Returns')
plt.show()

print("Each point is a 3-day window of returns.")
print("TDA analyzes the 'shape' of this cloud to extract features.")

## 4. Joining Features

Now we combine sentiment (aggregated to daily) with price and topology features.

In [ ]:
daily_sentiment = aggregate_daily_sentiment(scored)
X = join_sentiment_with_prices(daily_sentiment, price_features)

print(f"Final feature matrix: {X.shape}")
print(f"Date range: {X['date'].min()} to {X['date'].max()}")
print(f"Label balance: {X['label'].mean():.1%} positive")

X.tail()

In [ ]:
# Correlation with the target
feature_cols = ['ret_5', 'ret_10', 'vol_20', 'sent_mean', 'topo_d0_total', 'topo_entropy']
correlations = X[feature_cols + ['label']].corr()['label'].drop('label').sort_values()

plt.figure(figsize=(8, 4))
correlations.plot.barh()
plt.axvline(0, color='black', linewidth=0.5)
plt.xlabel('Correlation with next-day up')
plt.title('Feature Correlations')
plt.tight_layout()
plt.show()

## 5. Model Training

We train a simple logistic regression. The key detail is using a time-based split (not random) to avoid lookahead bias.

In [ ]:
model, report, threshold, features_used = train_model_from_frame(
    X, model_type='logistic', test_frac=0.2
)

print(f"Features used: {features_used}")
print(f"Threshold (tuned on train): {threshold:.2f}")
print(f"\nTest set results:")
print(f"  Accuracy:  {report.accuracy:.3f}")
print(f"  Precision: {report.precision:.3f}")
print(f"  Recall:    {report.recall:.3f}")
print(f"  AUC:       {report.auc:.3f}")

## 6. Backtesting

Simple strategy:
- Go long on tickers where predicted P(up) > threshold AND 5-day momentum is positive
- Equal weight across all positions
- Rebalance daily with 3bps round-trip cost

In [ ]:
# ML-based strategy
X_ml = add_ml_prob(X.copy(), model, features_used, out_col='p_up')
signals_ml = apply_signals(X_ml, threshold=threshold, require_mom_agree=True)
perf_ml, metrics_ml = backtest_equal_weight(signals_ml, cost_bps=3.0)

# Simple rule-based baseline
X_rule = add_quick_prob(X.copy())
signals_rule = apply_signals(X_rule, threshold=0.55, require_mom_agree=True)
perf_rule, metrics_rule = backtest_equal_weight(signals_rule, cost_bps=3.0)

print("Backtest Results")
print("="*45)
print(f"{'Strategy':<20} {'Sharpe':>8} {'Max DD':>10} {'Trades':>8}")
print("-"*45)
print(f"{'ML (Logistic)':<20} {metrics_ml['sharpe']:>8.2f} {metrics_ml['max_dd']:>9.2%} {metrics_ml['trades']:>8}")
print(f"{'Rule-based':<20} {metrics_rule['sharpe']:>8.2f} {metrics_rule['max_dd']:>9.2%} {metrics_rule['trades']:>8}")

In [ ]:
# Plot equity curves
plt.figure(figsize=(10, 5))

if not perf_ml.empty:
    plt.plot(pd.to_datetime(perf_ml['date']), perf_ml['equity'], 
             label=f"ML (Sharpe: {metrics_ml['sharpe']:.2f})", linewidth=1.5)

if not perf_rule.empty:
    plt.plot(pd.to_datetime(perf_rule['date']), perf_rule['equity'], 
             label=f"Rule-based (Sharpe: {metrics_rule['sharpe']:.2f})", linewidth=1.5, alpha=0.8)

plt.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Starting capital')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.title('Strategy Comparison')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Today's Signals

Based on the latest data, here are the current recommendations (if any meet the threshold).

In [ ]:
picks = todays_signals(X_ml, threshold=threshold, require_mom_agree=True)

if picks.empty:
    print("No signals today.")
    print(f"\nClosest candidates:")
    latest = X_ml[X_ml['date'] == X_ml['date'].max()]
    display(latest[['ticker', 'p_up', 'ret_5', 'sent_mean']].sort_values('p_up', ascending=False))
else:
    print(f"Signals for {picks['date'].iloc[0]}:")
    display(picks)

## Summary

This pipeline combines three types of features:
1. **Sentiment** - FinBERT scores on financial headlines
2. **Price momentum** - standard returns and volatility
3. **Topology** - persistent homology features from delay-embedded price series